# Azure Stay - CP372 Final Project
## Channel Profitability & Distribution Cost Analysis

โปรเจ็กต์นี้สอดคล้องกับโจทย์ final project เรื่อง **High Distribution Costs / Commission Trap**  
และต่อยอดจากงานเดิม **HW3_Hotel_Descriptive_Diagnostic** กับ **Project Canvas**

### เป้าหมาย
1. เปรียบเทียบ **Net ADR** ระหว่าง Direct กับ OTA  
2. เปรียบเทียบ **Cost %** ของ commission model แต่ละแบบ  
3. หาช่องทางและ rate code ที่มี **Cancellation Rate สูง**  
4. ประเมินประสิทธิภาพของ **Direct Marketing Spend**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

## 1) Load data

> ถ้ารันบน Colab ให้ upload โฟลเดอร์ `data/` หรือแก้ path ให้ตรงกับที่เก็บไฟล์


In [ ]:
DATA_PATH = '../data'

fact_bookings = pd.read_csv(f'{DATA_PATH}/fact_bookings.csv', parse_dates=['booking_date', 'check_in_date'])
dim_channels = pd.read_csv(f'{DATA_PATH}/dim_channels.csv')
dim_rate_codes = pd.read_csv(f'{DATA_PATH}/dim_rate_codes.csv')
fact_marketing_spend = pd.read_csv(f'{DATA_PATH}/fact_marketing_spend.csv', parse_dates=['spend_date'])

print(fact_bookings.shape, dim_channels.shape, dim_rate_codes.shape, fact_marketing_spend.shape)
fact_bookings.head()

## 2) Data model

In [ ]:
df = (
    fact_bookings
    .merge(dim_channels, on='channel_id', how='left')
    .merge(dim_rate_codes, on='rate_code_id', how='left')
)
df.head()

## 3) KPI 1: Net ADR by channel type

In [ ]:
non_cancel = df[df['status'] != 'Cancelled'].copy()

net_adr = (
    non_cancel.groupby('channel_type')
    .agg(
        bookings=('booking_id', 'count'),
        gross_room_revenue=('gross_room_revenue', 'sum'),
        commission_amount=('commission_amount', 'sum'),
        net_room_revenue=('net_room_revenue', 'sum')
    )
)

net_adr['net_adr'] = net_adr['net_room_revenue'] / net_adr['bookings']
net_adr = net_adr.sort_values('net_adr', ascending=False)
net_adr.round(2)

In [ ]:
ax = net_adr['net_adr'].plot(kind='bar')
ax.set_title('Net ADR by Channel Type')
ax.set_ylabel('Net ADR')
ax.set_xlabel('Channel Type')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4) KPI 2: Commission Cost % by commission model

In [ ]:
cost_pct = (
    non_cancel.groupby('commission_model')
    .agg(
        gross_room_revenue=('gross_room_revenue', 'sum'),
        commission_amount=('commission_amount', 'sum')
    )
)
cost_pct['cost_pct'] = (cost_pct['commission_amount'] / cost_pct['gross_room_revenue']) * 100
cost_pct.sort_values('cost_pct').round(2)

In [ ]:
ax = cost_pct['cost_pct'].sort_values().plot(kind='bar')
ax.set_title('Commission Cost % by Model')
ax.set_ylabel('Cost %')
ax.set_xlabel('Commission Model')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5) KPI 3: Cancellation analysis

In [ ]:
cancel_by_channel = (
    df.assign(is_cancelled=(df['status'] == 'Cancelled').astype(int))
      .groupby('channel_name')['is_cancelled']
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

cancel_by_rate = (
    df.assign(is_cancelled=(df['status'] == 'Cancelled').astype(int))
      .groupby('rate_name')['is_cancelled']
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

print('Cancellation Rate by Channel (%)')
display(cancel_by_channel.round(2).to_frame())

print('Cancellation Rate by Rate Code (%)')
display(cancel_by_rate.round(2).to_frame())

In [ ]:
ax = cancel_by_rate.plot(kind='bar')
ax.set_title('Cancellation Rate by Rate Code')
ax.set_ylabel('Cancellation Rate (%)')
ax.set_xlabel('Rate Code')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 6) KPI 4: Direct marketing efficiency

In [ ]:
direct_bookings = df[(df['channel_id'] == 'CH_01') & (df['status'] != 'Cancelled')].copy()

daily_direct_bookings = (
    direct_bookings.groupby(direct_bookings['booking_date'].dt.date)
    .size()
    .reset_index(name='direct_bookings')
)

daily_spend = (
    fact_marketing_spend.groupby(fact_marketing_spend['spend_date'].dt.date)
    .agg(cost_amount=('cost_amount', 'sum'), clicks=('clicks', 'sum'))
    .reset_index()
)

marketing = daily_spend.merge(
    daily_direct_bookings,
    left_on='spend_date',
    right_on='booking_date',
    how='left'
).drop(columns=['booking_date']).fillna({'direct_bookings': 0})

marketing['cost_per_booking'] = marketing['cost_amount'] / marketing['direct_bookings'].replace(0, np.nan)

marketing.describe().round(2)

In [ ]:
monthly_marketing = marketing.copy()
monthly_marketing['month'] = pd.to_datetime(monthly_marketing['spend_date']).dt.to_period('M').astype(str)

monthly_summary = (
    monthly_marketing.groupby('month')
    .agg(cost_amount=('cost_amount', 'sum'),
         clicks=('clicks', 'sum'),
         direct_bookings=('direct_bookings', 'sum'))
)
monthly_summary['cost_per_booking'] = monthly_summary['cost_amount'] / monthly_summary['direct_bookings'].replace(0, np.nan)
monthly_summary.tail(12).round(2)

In [ ]:
ax = monthly_summary['cost_per_booking'].dropna().plot()
ax.set_title('Monthly Cost per Direct Booking')
ax.set_ylabel('Cost per Booking')
ax.set_xlabel('Month')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7) Executive summary

In [ ]:
direct_net_adr = net_adr.loc['Direct', 'net_adr']
ota_net_adr = net_adr.loc['OTA', 'net_adr']
flat_fee_cost = cost_pct.loc['Flat Fee', 'cost_pct']
percentage_cost = cost_pct.loc['Percentage', 'cost_pct']
promo_cancel = cancel_by_rate.loc['Promotional Rate']
expedia_cancel = cancel_by_channel.loc['Expedia']

print(f'Direct Net ADR = {direct_net_adr:,.2f}')
print(f'OTA Net ADR = {ota_net_adr:,.2f}')
print(f'Direct advantage over OTA = {direct_net_adr - ota_net_adr:,.2f}')
print(f'Flat Fee Cost % = {flat_fee_cost:.2f}%')
print(f'Percentage Cost % = {percentage_cost:.2f}%')
print(f'Promotional Rate Cancellation = {promo_cancel:.2f}%')
print(f'Expedia Cancellation = {expedia_cancel:.2f}%')

## 8) Recommendations

### Recommendation 1: Shift mix from OTA to Direct / low-cost channels
- เพิ่มสัดส่วน Direct Website, Walk-in และ Corporate
- จำกัด inventory ใน OTA ที่ cost สูง โดยเฉพาะช่วง high demand

### Recommendation 2: Tighten promotional cancellation policy
- สำหรับ Promotional Rate ใช้ non-refundable หรือ deposit required
- ลด free cancellation ในช่วง demand สูง

### Recommendation 3: Optimize direct marketing before scaling spend
- ทำ A/B testing หน้า booking engine
- วัด conversion by campaign / platform
- เพิ่มสิทธิพิเศษ direct booking แทนการลดราคาหนัก
